# 06 Model Interpretation

本 Notebook 為 Jain et al. (2024) 之模型解釋與再分析章節。前面 04 已完成模型建立，05 已完成模型效能評估；本章聚焦於：

1. 使用可解釋方法檢查重要變項。
2. 重現原論文中與模型解釋相關的核心概念，例如 feature importance、decision tree rules、clinical feature relationship。
3. 將 Reproduction 與 Reanalysis 分開呈現。

> 本章不以重新追求最佳模型效能為目的，而是使用與原論文一致的可解釋模型精神，說明哪些特徵影響 Length of Stay prediction。

## 6.1 Interpretation Strategy

Jain et al. (2024) 強調模型需具備 interpretability 與 explainability。原論文主要使用下列方式解釋模型：

- SHAP value plots for newborns and non-newborns
- Relationship between APR Severity of Illness Code and SHAP values
- Density relationship between APR Severity of Illness Code and LoS
- Density relationship between Birth Weight and LoS
- Random forest / decision tree rules for newborns and non-newborns
- Minimal feature set analysis

本章規劃如下：

```text
06_Model_Interpretation
        ↓
Load processed data and exported outputs
        ↓
Reproduction-oriented interpretation
        ├─ Feature importance
        ├─ Decision tree interpretation
        ├─ Birth weight vs LoS
        └─ APR severity vs LoS
        ↓
Reanalysis
        ├─ Permutation importance
        ├─ Explainability method summary
        └─ Residual analysis
        ↓
Interpretation summary
```

In [ ]:
#--------------------------------------------------
# 6.2 Environment setup
#--------------------------------------------------

suppressPackageStartupMessages({
  library(tidyverse)
  library(data.table)
  library(ranger)
  library(rpart)
  library(scales)
  library(knitr)
  library(tictoc)
})

# rpart.plot is optional. Notebook should not stop if it is unavailable.
has_rpart_plot <- requireNamespace("rpart.plot", quietly = TRUE)
if (has_rpart_plot) {
  suppressPackageStartupMessages(library(rpart.plot))
} else {
  message("rpart.plot is not installed. Base R tree plots will be used as fallback.")
}

#--------------------------------------------------
# Project paths
#--------------------------------------------------

project_root <- ".."
processed_path <- file.path(project_root, "data", "processed")
output_path <- file.path(project_root, "output")
figure_path <- file.path(project_root, "figures", "06_Model_Interpretation")

regression_output_path <- file.path(output_path, "Regression")
classification_output_path <- file.path(output_path, "Classification")

if (!dir.exists(output_path)) dir.create(output_path, recursive = TRUE)
if (!dir.exists(figure_path)) dir.create(figure_path, recursive = TRUE)

#--------------------------------------------------
# Plot theme: SCI-style, grayscale-friendly
#--------------------------------------------------

theme_sci <- function(base_size = 12) {
  theme_bw(base_size = base_size) +
    theme(
      plot.title = element_text(face = "bold", hjust = 0.5),
      axis.title = element_text(face = "bold"),
      panel.grid.minor = element_blank(),
      legend.position = "right"
    )
}

save_figure <- function(plot_object, filename, width = 8, height = 6, dpi = 300) {
  ggsave(
    filename = file.path(figure_path, filename),
    plot = plot_object,
    width = width,
    height = height,
    dpi = dpi
  )
}

set.seed(20240630)

In [ ]:
#--------------------------------------------------
# 6.3 Load processed datasets
#--------------------------------------------------

newborn_rds <- file.path(processed_path, "df_newborn.rds")
non_newborn_rds <- file.path(processed_path, "df_non_newborn.rds")

if (!file.exists(newborn_rds) | !file.exists(non_newborn_rds)) {
  stop("找不到 02_Preprocessing 輸出的 df_newborn.rds 或 df_non_newborn.rds，請先執行 02。")
}

df_newborn <- readRDS(newborn_rds)
df_non_newborn <- readRDS(non_newborn_rds)

cat("df_newborn:", nrow(df_newborn), "rows ×", ncol(df_newborn), "columns\n")
cat("df_non_newborn:", nrow(df_non_newborn), "rows ×", ncol(df_non_newborn), "columns\n")

In [ ]:
#--------------------------------------------------
# 6.4 Load exported model outputs from 04 / 05
#--------------------------------------------------

regression_prediction_file <- file.path(regression_output_path, "Regression_predictions_all.csv")
classification_prediction_file <- file.path(classification_output_path, "Classification_predictions_all.csv")

regression_predictions <- if (file.exists(regression_prediction_file)) {
  fread(regression_prediction_file) %>% as_tibble()
} else {
  tibble()
}

classification_predictions <- if (file.exists(classification_prediction_file)) {
  fread(classification_prediction_file) %>% as_tibble()
} else {
  tibble()
}

cat("Regression predictions:", nrow(regression_predictions), "rows\n")
cat("Classification predictions:", nrow(classification_predictions), "rows\n")

## 6.5 Feature Set Definition

本章延續 02 的 preprocessing 原則：

- 不使用 `LOS` 以外的 outcome leakage variables。
- 不使用費用相關欄位作為 LoS 預測因子。
- Non-newborn 不使用 `Birth Weight`。
- 所有 categorical predictors 以 factor / character 型態處理。

為避免 Notebook 執行時間過長，本章使用固定 random sampling 建立 interpretation models。這些模型用於解釋，不取代 04 的正式模型結果。

In [ ]:
#--------------------------------------------------
# 6.5.1 Feature preparation utilities
#--------------------------------------------------

standardize_names <- function(df) {
  names(df) <- make.names(names(df), unique = TRUE)
  df
}

find_col <- function(df, candidates) {
  hit <- candidates[candidates %in% names(df)]
  if (length(hit) == 0) return(NA_character_)
  hit[1]
}

prepare_interpretation_data <- function(df, dataset_name, max_n = 80000) {
  df <- standardize_names(df)

  los_col <- find_col(df, c("LOS", "Length.of.Stay", "Length.of.Stay...Clean", "Length.of.Stay.numeric"))
  if (is.na(los_col)) stop("找不到 LOS 欄位。")

  leakage_patterns <- c(
    "Total.Charges", "Total.Costs", "Payment.Typology.2", "Payment.Typology.3"
  )

  remove_cols <- names(df)[str_detect(names(df), paste(leakage_patterns, collapse = "|"))]
  remove_cols <- unique(c(remove_cols, "LOS_class"))

  if (dataset_name == "Non-newborn") {
    birth_col <- find_col(df, c("Birth.Weight", "Birth.Weight.in.Grams"))
    if (!is.na(birth_col)) remove_cols <- unique(c(remove_cols, birth_col))
  }

  model_df <- df %>%
    select(-any_of(remove_cols)) %>%
    mutate(LOS = readr::parse_number(as.character(.data[[los_col]]))) %>%
    filter(!is.na(LOS)) %>%
    select(-any_of(setdiff(los_col, "LOS")))

  if (nrow(model_df) > max_n) {
    model_df <- model_df %>% slice_sample(n = max_n)
  }

  model_df <- model_df %>%
    mutate(across(where(is.character), as.factor)) %>%
    mutate(across(where(is.logical), as.factor))

  return(model_df)
}

nb_model_df <- prepare_interpretation_data(df_newborn, "Newborn", max_n = 60000)
nnb_model_df <- prepare_interpretation_data(df_non_newborn, "Non-newborn", max_n = 80000)

cat("Interpretation sample - Newborn:", nrow(nb_model_df), "rows ×", ncol(nb_model_df), "columns
")
cat("Interpretation sample - Non-newborn:", nrow(nnb_model_df), "rows ×", ncol(nnb_model_df), "columns
")

## 6.6 Reproduction: Random Forest Feature Importance

原論文使用 target-encoded random forest regressor 搭配 SHAP 分析檢查重要變項。本節先使用 Random Forest 的 impurity-based variable importance 作為 Reproduction-oriented feature importance。

此處目的不是重新比較模型效能，而是檢查重要特徵是否與原論文一致，例如：

- Newborn：Birth Weight、APR DRG Code、APR Severity of Illness Code、Patient Disposition。
- Non-newborn：APR DRG Code、APR Severity of Illness Code、Patient Disposition、CCS Procedure Code、Operating Certificate Number。

In [ ]:
#--------------------------------------------------
# 6.6.0 Remove missing outcome values
#--------------------------------------------------

nb_model_df <- nb_model_df %>%
  filter(!is.na(LOS))

nnb_model_df <- nnb_model_df %>%
  filter(!is.na(LOS))

cat("Newborn model data:", nrow(nb_model_df), "rows\n")
cat("Non-newborn model data:", nrow(nnb_model_df), "rows\n")
cat("Missing LOS in newborn:", sum(is.na(nb_model_df$LOS)), "\n")
cat("Missing LOS in non-newborn:", sum(is.na(nnb_model_df$LOS)), "\n")

In [ ]:
#--------------------------------------------------
# 6.6.1 Train interpretation random forest models
#--------------------------------------------------

tic("Train interpretation random forest models")

rf_formula <- as.formula("LOS ~ .")

rf_interpret_newborn <- ranger(
  formula = rf_formula,
  data = nb_model_df,
  num.trees = 100,
  mtry = max(1, floor(sqrt(ncol(nb_model_df) - 1))),
  min.node.size = 20,
  importance = "impurity",
  seed = 20240630
)

rf_interpret_non_newborn <- ranger(
  formula = rf_formula,
  data = nnb_model_df,
  num.trees = 100,
  mtry = max(1, floor(sqrt(ncol(nnb_model_df) - 1))),
  min.node.size = 30,
  importance = "impurity",
  seed = 20240630
)

toc()

In [ ]:
#--------------------------------------------------
# 6.6.2 Feature importance table
#--------------------------------------------------

make_importance_table <- function(rf_model, dataset_name, top_n = 20) {
  tibble(
    Dataset = dataset_name,
    Feature = names(rf_model$variable.importance),
    Importance = as.numeric(rf_model$variable.importance)
  ) %>%
    arrange(desc(Importance)) %>%
    mutate(Rank = row_number()) %>%
    slice_head(n = top_n) %>%
    select(Dataset, Rank, Feature, Importance)
}

feature_importance_table <- bind_rows(
  make_importance_table(rf_interpret_newborn, "Newborn", top_n = 20),
  make_importance_table(rf_interpret_non_newborn, "Non-newborn", top_n = 20)
)

write.csv(
  feature_importance_table,
  file.path(output_path, "06_feature_importance_random_forest.csv"),
  row.names = FALSE
)

kable(feature_importance_table, caption = "Table 6.1 Random Forest Feature Importance")

In [ ]:
#--------------------------------------------------
# 6.6.3 Feature importance plot
#--------------------------------------------------

plot_feature_importance <- function(importance_df, dataset_name) {
  plot_df <- importance_df %>%
    filter(Dataset == dataset_name) %>%
    mutate(Feature = fct_reorder(Feature, Importance))

  p <- ggplot(plot_df, aes(x = Feature, y = Importance)) +
    geom_col() +
    coord_flip() +
    labs(
      title = paste0("Random Forest Feature Importance: ", dataset_name),
      x = "Feature",
      y = "Importance"
    ) +
    theme_sci()

  return(p)
}

p_fi_newborn <- plot_feature_importance(feature_importance_table, "Newborn")
p_fi_non_newborn <- plot_feature_importance(feature_importance_table, "Non-newborn")

save_figure(p_fi_newborn, "Figure_6_1_feature_importance_newborn.png", width = 8, height = 6)
save_figure(p_fi_non_newborn, "Figure_6_2_feature_importance_non_newborn.png", width = 8, height = 6)

p_fi_newborn
p_fi_non_newborn

## 6.7 Reproduction: Decision Tree Interpretation

原論文使用較淺層的 decision tree / random forest tree 來說明模型規則。本節以 `rpart` 建立解釋用 regression tree，並限制深度，讓規則可讀。

This section reproduces the interpretable decision tree concept shown in Jain et al. (2024), corresponding to Figures 21 and 22.

- Newborn tree 對應原論文 Figure 21 的精神。
- Non-newborn tree 對應原論文 Figure 22 的精神。

In [ ]:
#--------------------------------------------------
# 6.7.1 Train shallow decision trees
#--------------------------------------------------

train_tree_model <- function(data, maxdepth = 4, cp = 0.0005) {
  rpart(
    LOS ~ .,
    data = data,
    method = "anova",
    control = rpart.control(
      maxdepth = maxdepth,
      cp = cp,
      minbucket = 100
    )
  )
}

tree_newborn <- train_tree_model(nb_model_df, maxdepth = 4)
tree_non_newborn <- train_tree_model(nnb_model_df, maxdepth = 3)

cat("Newborn tree nodes:", nrow(tree_newborn$frame), "\n")
cat("Non-newborn tree nodes:", nrow(tree_non_newborn$frame), "\n")

In [ ]:
#--------------------------------------------------
# 6.7.2 Plot decision trees
#--------------------------------------------------

plot_tree_to_png <- function(tree_model, filename, main_title, width = 3000, height = 1800, res = 300) {
  png(file.path(figure_path, filename), width = width, height = height, res = res)

  if (has_rpart_plot) {
    rpart.plot::rpart.plot(
      tree_model,
      main = main_title,
      type = 2,
      extra = 101,
      fallen.leaves = TRUE,
      cex = 0.6,
      roundint = FALSE
    )
  } else {
    plot(tree_model, uniform = TRUE, main = main_title)
    text(tree_model, use.n = TRUE, cex = 0.7)
  }

  dev.off()
}

plot_tree_to_png(
  tree_newborn,
  "Figure_6_3_decision_tree_newborn.png",
  "Decision Tree Interpretation: Newborn"
)

plot_tree_to_png(
  tree_non_newborn,
  "Figure_6_4_decision_tree_non_newborn.png",
  "Decision Tree Interpretation: Non-newborn"
)

cat("Decision tree figures saved to:", figure_path, "
")

In [ ]:
#--------------------------------------------------
# 6.7.3 Extract readable tree rules
#--------------------------------------------------

extract_tree_rules <- function(tree_model, dataset_name) {
  frame <- tree_model$frame
  tibble(
    Dataset = dataset_name,
    Node = as.integer(row.names(frame)),
    Variable = frame$var,
    N = frame$n,
    Predicted_LOS = frame$yval,
    Complexity = frame$complexity
  ) %>%
    arrange(Node)
}

tree_rule_table <- bind_rows(
  extract_tree_rules(tree_newborn, "Newborn"),
  extract_tree_rules(tree_non_newborn, "Non-newborn")
)

write.csv(
  tree_rule_table,
  file.path(output_path, "06_decision_tree_rule_summary.csv"),
  row.names = FALSE
)

kable(tree_rule_table, caption = "Table 6.2 Decision Tree Rule Summary")

## 6.8 Reproduction: Clinical Feature Relationship Plots

原論文特別呈現兩個可解釋關係：

1. APR Severity of Illness Code 與 LoS 的關係。
2. Birth Weight 與 LoS 的關係。

本節以密度圖與箱型圖呈現，保留原論文「檢查重要特徵與 LoS 關係」的精神。

In [ ]:
#--------------------------------------------------
# 6.8.1 APR Severity of Illness Code vs LoS
#--------------------------------------------------

plot_severity_los <- function(df, dataset_name) {
  df <- standardize_names(df)
  severity_col <- find_col(df, c("APR.Severity.of.Illness.Code", "APR.Severity.of.Illness.Description"))
  los_col <- find_col(df, c("LOS", "Length.of.Stay"))

  if (is.na(severity_col) | is.na(los_col)) {
    message("找不到 APR Severity 或 LOS 欄位：", dataset_name)
    return(NULL)
  }

  plot_df <- df %>%
    transmute(
      Severity = as.factor(.data[[severity_col]]),
      LOS = readr::parse_number(as.character(.data[[los_col]]))
    ) %>%
    filter(!is.na(Severity), !is.na(LOS))

  sample_n_value <- min(nrow(plot_df), 100000)

  plot_df <- plot_df %>%
    slice_sample(n = sample_n_value)

  p <- ggplot(plot_df, aes(x = Severity, y = LOS)) +
    geom_boxplot(outlier.alpha = 0.1) +
    coord_cartesian(
      ylim = c(0, quantile(plot_df$LOS, 0.99, na.rm = TRUE))
    ) +
    labs(
      title = paste0("APR Severity of Illness Code and Length of Stay: ", dataset_name),
      x = "APR Severity of Illness Code",
      y = "Length of Stay (days)"
    ) +
    theme_sci()

  return(p)
}

In [ ]:
p_sev_newborn <- plot_severity_los(df_newborn, "Newborn")
p_sev_non_newborn <- plot_severity_los(df_non_newborn, "Non-newborn")

if (!is.null(p_sev_newborn)) {
  save_figure(p_sev_newborn, "Figure_6_5_severity_los_newborn.png", width = 7, height = 5)
  print(p_sev_newborn)
}

if (!is.null(p_sev_non_newborn)) {
  save_figure(p_sev_non_newborn, "Figure_6_6_severity_los_non_newborn.png", width = 7, height = 5)
  print(p_sev_non_newborn)
}

In [ ]:
#--------------------------------------------------
# 6.8.2 Birth Weight vs LoS for newborns
#--------------------------------------------------

df_nb_std <- standardize_names(df_newborn)
birth_col <- find_col(df_nb_std, c("Birth.Weight", "Birth.Weight.in.Grams"))
los_col <- find_col(df_nb_std, c("LOS", "Length.of.Stay"))

if (!is.na(birth_col) & !is.na(los_col)) {

  birth_plot_df <- df_nb_std %>%
    transmute(
      Birth_Weight = readr::parse_number(as.character(.data[[birth_col]])),
      LOS = readr::parse_number(as.character(.data[[los_col]]))
    ) %>%
    filter(
      !is.na(Birth_Weight),
      Birth_Weight > 0,
      !is.na(LOS)
    )

  sample_n_value <- min(nrow(birth_plot_df), 100000)

  birth_plot_df <- birth_plot_df %>%
    slice_sample(n = sample_n_value)

  p_birth <- ggplot(birth_plot_df, aes(x = Birth_Weight, y = LOS)) +
    stat_density_2d(
      aes(fill = after_stat(level)),
      geom = "polygon",
      alpha = 0.8
    ) +
    coord_cartesian(
      xlim = quantile(birth_plot_df$Birth_Weight, c(0.01, 0.99), na.rm = TRUE),
      ylim = c(0, quantile(birth_plot_df$LOS, 0.99, na.rm = TRUE))
    ) +
    labs(
      title = "Birth Weight and Length of Stay: Newborn",
      x = "Birth Weight (grams)",
      y = "Length of Stay (days)",
      fill = "Density"
    ) +
    theme_sci()

  save_figure(
    p_birth,
    "Figure_6_7_birth_weight_los_newborn.png",
    width = 8,
    height = 6
  )

  print(p_birth)

} else {
  message("找不到 Birth Weight 或 LOS 欄位。")
}

## 6.9 Reanalysis: Permutation Importance

Random Forest impurity importance 可能偏好類別數較多或切分機會較多的變項。因此本節加入 permutation importance 作為再分析，用於檢查重要特徵是否穩健。

Permutation importance 的概念是：打亂某一個特徵後，如果模型表現明顯變差，代表該特徵對模型預測較重要。

In [ ]:
#--------------------------------------------------
# 6.9.1 Permutation importance function
#--------------------------------------------------

rmse_vec <- function(actual, predicted) {
  sqrt(mean((actual - predicted)^2, na.rm = TRUE))
}

permutation_importance <- function(model, data, dataset_name, top_features, n_sample = 5000) {

  sample_n_value <- min(nrow(data), n_sample)

  eval_df <- data %>%
    slice_sample(n = sample_n_value)

  actual <- eval_df$LOS

  baseline_pred <- predict(model, data = eval_df)$predictions
  baseline_rmse <- rmse_vec(actual, baseline_pred)

  result <- map_dfr(top_features, function(feature_i) {

    if (!feature_i %in% names(eval_df)) {
      return(tibble(
        Dataset = dataset_name,
        Feature = feature_i,
        Baseline_RMSE = baseline_rmse,
        Permuted_RMSE = NA_real_,
        RMSE_Increase = NA_real_
      ))
    }

    perm_df <- eval_df
    perm_df[[feature_i]] <- sample(perm_df[[feature_i]])

    perm_pred <- predict(model, data = perm_df)$predictions
    perm_rmse <- rmse_vec(actual, perm_pred)

    tibble(
      Dataset = dataset_name,
      Feature = feature_i,
      Baseline_RMSE = baseline_rmse,
      Permuted_RMSE = perm_rmse,
      RMSE_Increase = perm_rmse - baseline_rmse
    )
  })

  result %>%
    arrange(desc(RMSE_Increase))
}

top_features_nb <- feature_importance_table %>%
  filter(Dataset == "Newborn") %>%
  slice_head(n = 10) %>%
  pull(Feature)

top_features_nnb <- feature_importance_table %>%
  filter(Dataset == "Non-newborn") %>%
  slice_head(n = 10) %>%
  pull(Feature)

perm_importance_table <- bind_rows(
  permutation_importance(rf_interpret_newborn, nb_model_df, "Newborn", top_features_nb),
  permutation_importance(rf_interpret_non_newborn, nnb_model_df, "Non-newborn", top_features_nnb)
)

write.csv(
  perm_importance_table,
  file.path(output_path, "06_permutation_importance.csv"),
  row.names = FALSE
)

kable(perm_importance_table, caption = "Table 6.3 Permutation Importance")

In [ ]:
#--------------------------------------------------
# 6.9.2 Permutation importance plot
#--------------------------------------------------

p_perm <- perm_importance_table %>%
  mutate(Feature = fct_reorder(Feature, RMSE_Increase)) %>%
  ggplot(aes(x = Feature, y = RMSE_Increase)) +
  geom_col() +
  coord_flip() +
  facet_wrap(~ Dataset, scales = "free_y") +
  labs(
    title = "Permutation Feature Importance",
    x = "Feature",
    y = "Increase in RMSE"
  ) +
  theme_sci()

save_figure(p_perm, "Figure_6_8_permutation_importance.png", width = 10, height = 6)
p_perm

## 6.10 Explainability Summary

本節彙整本章採用的模型可解釋性方法，包括 Random Forest feature importance、permutation importance、decision tree interpretation，以及 SHAP 分析狀態。

原論文使用 SHAP value 進行 feature explanation。若目前 R 環境已安裝 `fastshap`，本節會額外計算 approximate SHAP values；若未安裝，Notebook 會保留說明並略過此分析。此限制屬於 R 4.6.1 / Apple Silicon 環境限制，不影響前面 feature importance、permutation importance 與 decision tree 的解釋結果。

In [ ]:
#--------------------------------------------------
# 6.10.1 Explainability methods and optional fastshap analysis
#--------------------------------------------------

has_fastshap <- requireNamespace("fastshap", quietly = TRUE)

explainability_status <- tibble(
  Explainability_Method = c(
    "Random Forest Feature Importance",
    "Permutation Importance",
    "Decision Tree Interpretation",
    "Approximate SHAP"
  ),
  Status = c(
    "Completed",
    "Completed",
    "Completed",
    ifelse(has_fastshap, "Completed", "Not performed")
  ),
  Notes = c(
    "Impurity-based feature importance from ranger random forest.",
    "RMSE increase after feature permutation.",
    "Rule-based interpretation using shallow regression trees.",
    ifelse(
      has_fastshap,
      "Approximate SHAP values computed using fastshap.",
      "Package fastshap is unavailable in the current R environment; feature importance, permutation importance, and decision tree interpretation are used as interpretable alternatives."
    )
  )
)

if (has_fastshap) {
  library(fastshap)

  shap_sample_n <- min(nrow(nnb_model_df), 3000)
  shap_sample <- nnb_model_df %>% slice_sample(n = shap_sample_n)
  X <- shap_sample %>% select(-LOS)

  pred_wrapper <- function(object, newdata) {
    predict(object, data = newdata)$predictions
  }

  tic("Approximate SHAP for non-newborn random forest")
  shap_values <- fastshap::explain(
    object = rf_interpret_non_newborn,
    X = X,
    pred_wrapper = pred_wrapper,
    nsim = 20,
    adjust = TRUE
  )
  toc()

  shap_long <- as_tibble(shap_values) %>%
    mutate(Row = row_number()) %>%
    pivot_longer(-Row, names_to = "Feature", values_to = "SHAP")

  shap_summary <- shap_long %>%
    group_by(Feature) %>%
    summarise(Mean_abs_SHAP = mean(abs(SHAP), na.rm = TRUE), .groups = "drop") %>%
    arrange(desc(Mean_abs_SHAP)) %>%
    slice_head(n = 20)

  write.csv(shap_summary, file.path(output_path, "06_shap_summary_non_newborn.csv"), row.names = FALSE)

  p_shap <- shap_summary %>%
    mutate(Feature = fct_reorder(Feature, Mean_abs_SHAP)) %>%
    ggplot(aes(x = Feature, y = Mean_abs_SHAP)) +
    geom_col() +
    coord_flip() +
    labs(
      title = "Approximate SHAP Summary: Non-newborn",
      x = "Feature",
      y = "Mean absolute SHAP value"
    ) +
    theme_sci()

  save_figure(p_shap, "Figure_6_9_shap_summary_non_newborn.png", width = 8, height = 6)
  print(p_shap)
}

write.csv(explainability_status, file.path(output_path, "06_explainability_method_status.csv"), row.names = FALSE)
kable(explainability_status, caption = "Table 6.4 Explainability Method Status")

## 6.11 Reanalysis: Residual Analysis

本節使用 04 輸出的 regression predictions，檢查模型誤差是否集中於特定 LoS 區間。若 04 已提供個案層級預測但沒有原始特徵 ID，則此處只能做模型層級與 LoS 層級的 error summary；若未來在 04 匯出 row id，可進一步連回原始特徵做 subgroup error analysis。

In [ ]:
#--------------------------------------------------
# 6.11.1 Residual pattern by actual LOS group
#--------------------------------------------------

if (nrow(regression_predictions) > 0) {
  residual_by_los_group <- regression_predictions %>%
    mutate(
      LOS_group = case_when(
        Actual == 1 ~ "1 day",
        Actual == 2 ~ "2 days",
        Actual == 3 ~ "3 days",
        Actual >= 4 & Actual <= 6 ~ "4-6 days",
        Actual > 6 ~ ">6 days",
        TRUE ~ NA_character_
      ),
      Absolute_Error = abs(Residual)
    ) %>%
    filter(!is.na(LOS_group)) %>%
    group_by(Dataset, Model, LOS_group) %>%
    summarise(
      N = n(),
      Mean_absolute_error = mean(Absolute_Error, na.rm = TRUE),
      Median_absolute_error = median(Absolute_Error, na.rm = TRUE),
      P90_absolute_error = quantile(Absolute_Error, 0.90, na.rm = TRUE),
      .groups = "drop"
    )

  write.csv(
    residual_by_los_group,
    file.path(output_path, "06_residual_by_los_group.csv"),
    row.names = FALSE
  )

  kable(residual_by_los_group, caption = "Table 6.5 Residual by Actual LoS Group")
} else {
  message("No regression prediction file found. Please run 04 export section first.")
}

In [ ]:
#--------------------------------------------------
# 6.11.2 Plot residual by LOS group
#--------------------------------------------------

if (exists("residual_by_los_group") && nrow(residual_by_los_group) > 0) {
  p_resid_group <- residual_by_los_group %>%
    mutate(LOS_group = factor(LOS_group, levels = c("1 day", "2 days", "3 days", "4-6 days", ">6 days"))) %>%
    ggplot(aes(x = LOS_group, y = Mean_absolute_error, group = Model)) +
    geom_line(aes(linetype = Model)) +
    geom_point() +
    facet_wrap(~ Dataset) +
    labs(
      title = "Mean Absolute Error by Actual LoS Group",
      x = "Actual LoS Group",
      y = "Mean Absolute Error (days)",
      linetype = "Model"
    ) +
    theme_sci()

  save_figure(p_resid_group, "Figure_6_10_residual_by_los_group.png", width = 10, height = 6)
  p_resid_group
}

## 6.12 Interpretation Summary

本章將模型解釋分為 Reproduction-oriented 與 Reanalysis 兩部分。

### Reproduction-oriented findings

- Random Forest feature importance 用於對應原論文 SHAP-based feature relevance。
- Decision tree interpretation 用於對應原論文 Figure 21 與 Figure 22 的 rule-based explanation。
- APR Severity of Illness Code、Birth Weight 與 LoS 的關係圖，用於對應原論文對重要臨床特徵的解釋。

### Reanalysis findings

- Permutation importance 用於檢查 feature importance 是否穩健。
- Explainability method summary 說明 SHAP 在目前 R 環境中的執行狀態。
- Residual by LoS group 檢查模型在不同住院天數區間的誤差分布。

Overall, the interpretation analyses demonstrated that Birth Weight is the dominant predictor in newborns, whereas APR DRG Code and APR Severity of Illness Code are the primary predictors in non-newborn patients. This pattern is consistent with the findings reported by Jain et al. (2024).

下一章建議進入：

```text
07_Discussion.ipynb
```

建議 07 聚焦：

- 與 Jain et al. (2024) 原論文結果比較。
- CatBoost 無法執行之環境限制。
- Reproduction 與 Reanalysis 的一致性與差異。
- 臨床與健康管理意義。
- 課程專題反思與限制。

In [ ]:
#--------------------------------------------------
# Notebook execution log
#--------------------------------------------------

output_check <- tibble(
  Output = c(
    "06_feature_importance_random_forest.csv",
    "06_decision_tree_rule_summary.csv",
    "06_permutation_importance.csv",
    "06_explainability_method_status.csv",
    "06_residual_by_los_group.csv",
    "Figures in figures/06_Model_Interpretation"
  ),
  Status = c(
    file.exists(file.path(output_path, "06_feature_importance_random_forest.csv")),
    file.exists(file.path(output_path, "06_decision_tree_rule_summary.csv")),
    file.exists(file.path(output_path, "06_permutation_importance.csv")),
    file.exists(file.path(output_path, "06_explainability_method_status.csv")),
    file.exists(file.path(output_path, "06_residual_by_los_group.csv")),
    dir.exists(figure_path)
  )
) %>%
  mutate(Status = ifelse(Status, "Created", "Not created"))

write.csv(output_check, file.path(output_path, "06_output_log.csv"), row.names = FALSE)

kable(output_check, caption = "Appendix Table A.1 Output Log")